# 01 Characterisation

This notebook characterises how RPF/sign-error labels appear in final Alpha and Beta.

**Inputs.** It reads `dataset/final/dataset_alpha.parquet` and `dataset/final/dataset_beta.parquet` created by Notebook 0.

**Outputs.** It writes occurrence summaries, temporal summaries, event-duration summaries, and three paper-facing characterisation figures under `outputs/*/01_characterisation/`.

**Key decisions.** Gamma is intentionally excluded because it is a one-site forecast-impact case study, not a population characterisation dataset. The notebook uses the final-layer `alpha_*` and `beta_*` site IDs.

**When to rerun.** Rerun after Notebook 0 whenever Alpha/Beta final datasets change, especially after replacing provisional Beta labels with the manually reviewed oracle labels.


## 1. Imports And Paths

Load the v2 config, resolve output folders, and confirm that the notebook reads from `dataset/final/`. The plotting style and all summary logic come from `_experiment_helpers.py` so figure formatting stays consistent across notebooks.


In [ ]:
from pathlib import Path
import sys

# Keep notebook imports stable whether the notebook is run from JupyterLab,
# VS Code, or the repository root.
article_root = Path.cwd()
while article_root.name != "2_journal_article":
    if article_root.parent == article_root:
        raise RuntimeError("Could not locate publication/2_journal_article")
    article_root = article_root.parent
notebook_dir = article_root / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import _experiment_helpers as h

cfg = h.load_config(article_root)
paths = h.article_paths(article_root, cfg)
h.ensure_output_dirs(paths)
print(f"Article root: {article_root}")
print(f"Config schema: {cfg['schema_version']}")
print(f"Output root: {paths.outputs}")

print(article_root / cfg["paths"]["alpha_dataset_path"])
print(article_root / cfg["paths"]["beta_dataset_path"])


## 2. Load Final Alpha And Beta

`h.load_dataset()` validates the seven-column schema, parses timestamps as dataset wall-clock time, recomputes day labels from interval labels, checks duplicates, and derives analysis-only columns such as `reference_net_load_MW`, month, hour, weekday, and season. These derived columns are not written back into the final datasets.


In [ ]:
# Load final datasets with validation and derived analysis columns.
alpha = h.load_dataset(article_root, cfg, "alpha")
beta = h.load_dataset(article_root, cfg, "beta")
[h.dataset_summary(alpha, "Alpha"), h.dataset_summary(beta, "Beta")]


## 3. Build Characterisation Outputs

`h.run_characterisation()` creates the full characterisation artifact set. Internally it computes site-level RPF occurrence, month/hour temporal summaries, contiguous RPF event durations, compact table summaries, and journal-style figures. The intermediate CSVs are intentionally more detailed than the final paper tables so unusual patterns can be debugged later.


In [ ]:
# Write intermediates, compact tables, figures, and a manifest for Notebook 1.
outputs = h.run_characterisation(article_root)
outputs["occurrence_dataset"]


## 4. Inspect Important Tables

Use these previews to sanity-check the story before moving to correction validation. Check whether the highest-RPF sites make intuitive sense, whether event durations are plausible, and whether any single site dominates the results unexpectedly.


In [ ]:
display(outputs["occurrence"].sort_values(["dataset", "rpf_days"], ascending=[True, False]).head(20))
display(outputs["event_summary"])
display(outputs["events"].sort_values("duration_minutes", ascending=False).head(10))
